# Real-World Data Backtesting Demo

This demo showcases a **professional quantitative workflow**:
1. **Data Fetching**: Download real market data from Yahoo Finance.
2. **Walk-Forward Validation**: Find robust parameters that avoid overfitting.
3. **Final Backtest**: Run with validated parameters and realistic costs (slippage + commissions).
4. **Results Analysis**: Equity curve and performance metrics.

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_fetcher import DataFetcher
from src.data_handler import DataHandler
from src.strategy import MovingAverageCrossStrategy
from src.portfolio import Portfolio
from src.engine import BacktestEngine
from src.performance import create_equity_curve, calculate_drawdown, calculate_sharpe_ratio
from src.visualization import plot_equity_curve, plot_drawdown
from src.walk_forward import WalkForwardAnalyzer
from src.config import setup_logging, SLIPPAGE_RATE, COMMISSION_RATE

setup_logging()
print(f"Execution Costs: Slippage={SLIPPAGE_RATE*100:.2f}%, Commission={COMMISSION_RATE*100:.1f}%")

---
## 1. Fetch Real Market Data
Download Apple (AAPL) data. Data is cached locally after first download.

In [ ]:
fetcher = DataFetcher()
symbol = "AAPL"
start = "2023-01-01"
end = "2024-01-01"

csv_path = fetcher.get_data(symbol, start, end)
print(f"Data loaded: {csv_path}")

---
## 2. Walk-Forward Validation
**Before** running a backtest with fixed parameters, we validate which parameters are robust across different market regimes.

This prevents **overfitting** — finding parameters that only worked in the past but fail in the future.

In [ ]:
# Define parameter grid to explore
param_grid = {
    'short_window': [10, 20, 30],
    'long_window': [40, 50, 60]
}

# Walk-Forward: 120 days in-sample (optimize), 40 days out-of-sample (validate)
wfa = WalkForwardAnalyzer(
    data_path=csv_path,
    symbol=symbol,
    strategy_cls=MovingAverageCrossStrategy,
    param_grid=param_grid,
    in_sample_periods=120,
    out_of_sample_periods=40
)

wfa_results = wfa.run()
wfa_results

In [ ]:
# Get the most robust parameters (most frequently selected across windows)
best_params = wfa.get_recommended_params(wfa_results)
print(f"\n✓ Recommended Parameters: {best_params}")

# WFA Performance Summary
print(f"\n=== Walk-Forward OOS Summary ===")
print(f"Mean OOS Return: {wfa_results['OOS_Total_Return'].mean()*100:.2f}%")
print(f"Mean OOS Sharpe: {wfa_results['OOS_Sharpe_Ratio'].mean():.2f}")

---
## 3. Final Backtest with Validated Parameters
Now we run a full backtest using the **WFA-recommended parameters** — not arbitrary ones.

In [ ]:
# Use validated parameters from WFA
short_w = int(best_params['short_window'])
long_w = int(best_params['long_window'])

print(f"Running final backtest with: short_window={short_w}, long_window={long_w}")

data_handler = DataHandler(csv_path, symbol)
strategy = MovingAverageCrossStrategy(short_window=short_w, long_window=long_w)
portfolio = Portfolio(initial_capital=100000)

engine = BacktestEngine(data_handler, strategy, portfolio)
engine.run()

---
## 4. Results Analysis
Equity curve and performance metrics from the validated backtest.

In [ ]:
equity_df = create_equity_curve(portfolio.history)

if not equity_df.empty:
    # Calculate metrics
    initial = equity_df['equity'].iloc[0]
    final = equity_df['equity'].iloc[-1]
    total_return = (final - initial) / initial
    sharpe = calculate_sharpe_ratio(equity_df)
    dd = calculate_drawdown(equity_df)
    max_dd = dd.min()
    
    print(f"=== Final Backtest Results ===")
    print(f"Total Return: {total_return*100:.2f}%")
    print(f"Sharpe Ratio: {sharpe:.2f}")
    print(f"Max Drawdown: {max_dd*100:.2f}%")
    
    # Plot
    plot_equity_curve(equity_df, title=f"{symbol} Backtest (WFA-Validated: {short_w}/{long_w})")
    plot_drawdown(dd, title="Strategy Drawdown")
else:
    print("No trades executed.")

---
## 5. Conclusion
This workflow demonstrates:
- **Data-driven parameter selection** via Walk-Forward Analysis
- **Realistic execution costs** with slippage and commissions
- **Proper validation** to avoid overfitting

Check `logs/backtest.log` for detailed execution logs.